# Position decoding: InSeq and OutSeq

Uses shared preprocessing from `lit-review` commit `987145627a25fe4e7ddc0f5f29b04dc3119420e7`.
The new flag **must be enabled**: the decoding profile remains InSeq-only by default.

Generate a NEW export with the shared preprocessing package available on PYTHONPATH:
```sh
python -m hippocampal_preprocessing --profile inseq-decoding --include-outseq --raw-root /path/to/raw_data --output data/processed/position_inseq_outseq_9871456 --no-plots
```
Set `OST_POSITION_EXPORTS` to a different export directory if necessary. The loader rejects old InSeq-only configurations and verifies aligned position, odor and trial-type labels.

Keep 0–500 ms at 1 kHz and 60 Hz notch for this update, so trial inclusion is the intended change. Both correct and incorrect trials are eligible; early withdrawal and quality exclusions still apply. Shorter windows can retain more OutSeq trials, but need their own regenerated export and matched modeling configuration.

Raw and CEBRA use identical retained trials and chronological source splits (70/15/15). All learned scaling, quality thresholds and encoder fitting use training data only. Filter supports cannot cross splits. Inferred position-1 groups crossing splits are removed; these are not verified sequence IDs. Upstream training statistics stay frozen after this conservative group purge.

CEBRA: 8 dimensions, 10 ms receptive field and positive lag, 1,000 updates; sampling never crosses trial boundaries. Both representations use ten 50 ms bins. Logistic regression uses validation-selected L1 strength (SAGA solver) and balanced class weights; kNN tunes neighbors and weights on validation only.

Train on both trial types with position targets. Report overall, InSeq and OutSeq performance separately. Saved predictions include odor and position, allowing comparison of position matches against odor-associated position matches on OutSeq. Sparse OutSeq subsets may omit classes; subgroup balanced accuracy averages only present classes and reports class coverage. This is descriptive evidence, not proof of independent position coding. Consider withdrawal duration and other behavioral differences as possible confounds.

L1 is chosen for sparse coefficient inspection. Correlated features may exchange weights; coefficients are associations, not causal importance. A latent dimension is learned, not a named physiological source. Sparsity alone is not evidence of stable interpretation. Validation picks C; test results must not guide choices.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import os
import sys
from IPython.display import display

repo = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'src' / 'position_modeling.py').is_file())
sys.path.insert(0, str(repo / 'src'))
from position_modeling import run

raw_root = Path(os.environ.get('OST_POSITION_EXPORTS', repo / 'data' / 'processed' / 'position_inseq_outseq_9871456'))
output = repo / 'results' / ('position_modeling_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
metrics = run(raw_root, output)
comparison = metrics.pivot(index='rat', columns=['decoder', 'representation'], values='test_balanced_accuracy')
display(comparison.round(3))
display(comparison.mean().round(3))

import pandas as pd
display(pd.read_csv(output / 'trial_counts.csv'))
display(pd.read_csv(output / 'subgroup_metrics.csv').round(3))


# L1 run, November 10, 2026

Mean five-rat test balanced accuracy:

| Decoder | Raw | CEBRA |
|---|---:|---:|
| logistic | 33.1% | 26.4% |
| knn | 32.5% | 25.2% |

Same 114 test trials across all methods, including 15 OutSeq. These are exploratory results. L1 feature selection requires stability analysis before biological interpretation. Three decoder-contract tests passed; all five rats completed without convergence failures.

Results: `results/position_l1_verified_20261110`. L2 results are preserved in `results/archive/position_outseq_9871456_20261110`. Full provenance, coefficients and fitted models are in the L1 directory.
